# Solar Filament Segmentation 2026 — Reproduce, Train, Tune, Submit

**Competition dataset:** attach the official `filament-segmentation-2026` dataset. **Notebook settings:** Accelerator `GPU` (T4/P100), Internet `On` (needed only to clone the public GitHub source and optionally install missing packages).

This notebook does **not** claim a winning score. It trains a new model from official training images, calibrates against held-out observation dates, predicts every test image and emits `/kaggle/working/submission.csv`. No Kaggle test ground truth, private leaderboards, or unauthorized external labels are used. Execution can take hours.


In [ ]:
import sys, subprocess, os, pathlib, importlib.util, torch
print("Python:",sys.version,"Torch:",torch.__version__,"CUDA:",torch.cuda.is_available())
assert torch.cuda.is_available(), "Enable a GPU accelerator in Kaggle Notebook settings."
assert not pathlib.Path('/kaggle/working/Solar-Filament-Segmentation').exists(), "Restart the session for a clean reproduction."
subprocess.run(['git','clone','--depth','1','https://github.com/ahpa5246-lgtm/Solar-Filament-Segmentation.git','/kaggle/working/Solar-Filament-Segmentation'],check=True)
subprocess.run([sys.executable,'-m','pip','install','-q','pycocotools==2.0.8'],check=True)
subprocess.run([sys.executable,'-m','pip','install','-q','--no-deps','-e','/kaggle/working/Solar-Filament-Segmentation'],check=True)
print('Installed and cloned source code.')


## Step 1 — Verify data and run tests
No training or prediction begins unless the official MAGFiLO JSON and train/test JPEG directories are actually attached. Kaggle images are 2048×2048 grayscale.


In [ ]:
from solar.data import discover
a,train_dir,test_dir=discover('/kaggle/input')
print('Annotations:',a,'Training images:',train_dir,'Test images:',test_dir)
subprocess.run([sys.executable,'-m','pytest','-q','/kaggle/working/Solar-Filament-Segmentation/tests'],check=True)
subprocess.run([sys.executable,'-m','solar.audit','--data','/kaggle/input'],check=True)


## Step 2 — Train with a date-grouped validation holdout
Default training hyperparameters are hypotheses, not competition-validated claims. To change GPU memory, set `BATCH=2`. Allow training to finish and produce `best.pt`. **Never run inference with untrained weights.**


In [ ]:
BATCH=4
EPOCHS=30
STEPS=160
OUT='/kaggle/working/solar_artifacts'
subprocess.run([sys.executable,'-m','solar.train','--data','/kaggle/input','--out',OUT,'--epochs',str(EPOCHS),'--steps-per-epoch',str(STEPS),'--batch',str(BATCH),'--seed','42'],check=True)
assert pathlib.Path(OUT,'best.pt').is_file()


## Step 3 — Select thresholds by held-out Panoptic Quality
This calibration examines only training annotations held out *by date*, never test labels. Each source image may have multiple separate human segmentations; all are assessed. Scores measured here are **not** Kaggle leaderboard results.


In [ ]:
subprocess.run([sys.executable,'-m','solar.tune','--data','/kaggle/input','--checkpoint',f'{OUT}/best.pt','--out',OUT,'--max-images','16','--batch',str(BATCH)],check=True)
import json
print('Chosen params:',json.load(open(f'{OUT}/best_params.json')))
print('Top validation PQ:',json.load(open(f'{OUT}/tuning_results.json'))[0]['pq'])


## Step 4 — Predict every test image and generate the submission
The entire test directory is processed, without a debug image limit. Optional TTA averages original and mirrored predictions. Masks are encoded as official compressed COCO RLE strings.


In [ ]:
subprocess.run([sys.executable,'-m','solar.predict','--data','/kaggle/input','--checkpoint',f'{OUT}/best.pt','--params',f'{OUT}/best_params.json','--out','/kaggle/working/submission.csv','--batch',str(BATCH),'--tta'],check=True)
import pandas as pd
s=pd.read_csv('/kaggle/working/submission.csv',dtype=str)
assert list(s.columns)==['filament_id','segmentation_rle']
assert s['filament_id'].is_unique and s['segmentation_rle'].notna().all()
print('Submission rows:',len(s),'Distinct test observations with detections:',s['filament_id'].str.rsplit('_',n=1).str[0].nunique())
print('Ready for manual Kaggle Submit Prediction:',pathlib.Path('/kaggle/working/submission.csv'))


## Final evaluation beyond a single Kaggle upload
Save the trained checkpoint, `best_params.json`, date split manifest, `history.json`, `tuning_results.json`, full environment (`pip freeze`), and reproduce results in the organizer's four-page technical report. **The final judging also requires the public source repository, report PDF, and Google form by 15 Nov 2026, 11:59 PM CDT.** The Kaggle public score only reflects a subset of test images.


In [ ]:
subprocess.run([sys.executable,'-m','pip','freeze'],stdout=open(f'{OUT}/environment_frozen.txt','w'),check=True)
print('All supporting artifacts at',OUT)
